# 🧪 Lab 2: Fine-Tuning Gemma-4-31B (QLoRA)

In this lab, we will load the massive 31B model in 4-bit memory and attach small trainable "adapters" (LoRA) to teach it new things without needing a supercomputer.

In [ ]:
!pip install transformers accelerate bitsandbytes peft trl datasets

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "../gemma-4-31B"

# 1. Load in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

model = prepare_model_for_kbit_training(model)

In [ ]:
# 2. Setup LoRA Adapters
config = LoraConfig(
    r=8, 
    lora_alpha=32, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters() 
# You will see we only train ~0.1% of the model!

In [ ]:
# 3. From here, you would pass 'model' to the SFTTrainer from the 'trl' library 
# along with a HuggingFace dataset to begin fine-tuning!
print("Model is ready for training!")